# Multi-Rule Forgetting + Hyperparameter Tuning (Point 4)

Loads `forgetting_{rule}.json` for all four rules and:
1. Heatmap: crossing-point window length vs (threshold, lr) per rule
2. Heatmap: best simultaneous margin vs (threshold, lr) per rule
3. Summary bar: best config per rule
4. Re-run gap experiment with tuned hyperparams (see instructions at bottom)

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

RULES = ['current', 'approximate', 'long_warmup', 'ep']
FILES = {r: f'forgetting_{r}.json' for r in RULES}

data = {}
for rule, path in FILES.items():
    if os.path.exists(path):
        with open(path) as f:
            data[rule] = json.load(f)
        print(f'{rule}: loaded ({len(data[rule]["grid_results"])} configs)')
    else:
        print(f'{rule}: NOT FOUND — run forgetting_multi_rule.py --rule {rule}')

In [ ]:
def make_grid(grid_results, metric):
    thresholds = sorted(set(r['threshold'] for r in grid_results))
    lrs = sorted(set(r['learning_rate'] for r in grid_results))
    mat = np.full((len(thresholds), len(lrs)), np.nan)
    for r in grid_results:
        i = thresholds.index(r['threshold'])
        j = lrs.index(r['learning_rate'])
        mat[i, j] = r[metric]
    return mat, thresholds, lrs

def best_config(grid_results):
    valid = [r for r in grid_results if not np.isnan(r['mean_best_sim_margin'])]
    if valid:
        return max(valid, key=lambda r: r['mean_best_sim_margin'])
    return max(grid_results, key=lambda r: r['mean_n_both_correct'])

In [ ]:
# Fig 1: Heatmap of crossing window length per rule
n_rules = len(data)
if n_rules == 0:
    print('No data loaded yet.')
else:
    fig, axes = plt.subplots(1, n_rules, figsize=(5 * n_rules, 4))
    if n_rules == 1: axes = [axes]
    for ax, (rule, d) in zip(axes, data.items()):
        mat, thresholds, lrs = make_grid(d['grid_results'], 'mean_n_both_correct')
        im = ax.imshow(mat, cmap='YlGn', aspect='auto', origin='lower')
        ax.set_xticks(range(len(lrs))); ax.set_xticklabels([f'{lr:.3f}' for lr in lrs], rotation=45)
        ax.set_yticks(range(len(thresholds))); ax.set_yticklabels(thresholds)
        ax.set_xlabel('Learning rate'); ax.set_ylabel('Threshold')
        ax.set_title(f'{rule}\ncrossing window length')
        plt.colorbar(im, ax=ax)
    fig.suptitle('Mean # steps both correct during B-phase', fontsize=12)
    plt.tight_layout()
    plt.savefig('forgetting_tuning_fig1_window.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Fig 2: Heatmap of best simultaneous margin
if data:
    fig, axes = plt.subplots(1, n_rules, figsize=(5 * n_rules, 4))
    if n_rules == 1: axes = [axes]
    for ax, (rule, d) in zip(axes, data.items()):
        mat, thresholds, lrs = make_grid(d['grid_results'], 'mean_best_sim_margin')
        vmax = np.nanmax(np.abs(mat)) if not np.all(np.isnan(mat)) else 1.0
        im = ax.imshow(mat, cmap='RdYlGn', vmin=-vmax, vmax=vmax,
                       aspect='auto', origin='lower')
        ax.set_xticks(range(len(lrs))); ax.set_xticklabels([f'{lr:.3f}' for lr in lrs], rotation=45)
        ax.set_yticks(range(len(thresholds))); ax.set_yticklabels(thresholds)
        ax.set_xlabel('Learning rate'); ax.set_ylabel('Threshold')
        ax.set_title(f'{rule}\nbest sim. margin')
        plt.colorbar(im, ax=ax)
    fig.suptitle('Best simultaneous margin at crossing point (max over steps of min(margin_A, margin_B))', fontsize=11)
    plt.tight_layout()
    plt.savefig('forgetting_tuning_fig2_margin.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Fig 3: Best config per rule — summary table
print('\n=== Best hyperparams per rule ===')
best_configs = {}
for rule, d in data.items():
    best = best_config(d['grid_results'])
    best_configs[rule] = best
    print(f'{rule:15s}  threshold={best["threshold"]}  lr={best["learning_rate"]}  '
          f'window={best["mean_n_both_correct"]:.1f}  sim_margin={best["mean_best_sim_margin"]:.3f}')

print('\n=== Commands to re-run gap experiment with tuned hyperparams ===')
print('Run from experiments3/ directory:')
for rule, best in best_configs.items():
    print(f'  python gap_experiment.py --rules {rule} '
          f'--threshold {best["threshold"]} --learning-rate {best["learning_rate"]} '
          f'--n-updates 60 --n-seeds 3 '
          f'--output gap_tuned_{rule}.json')